<a href="https://colab.research.google.com/github/tristanengst/fintechCS9832025fa/blob/main/2Analyze_financial_reports_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#CASE2 Analyze financial reports

**Purpose**:

This notebook is to analyze financial reports, particularly annual reports (10-K), using large language models (LLMs) such as GPT-4 or locally deployed models, and to generate comprehensive PDF reports that provide insights into a company’s financial health and performance over the fiscal year.



---
## Instructions

**Step 1 – Set runtime to L4 GPU**  
This notebook runs the LLM locally with [Ollama](https://ollama.com), so a GPU **is required**.  

1. Go to **Runtime → Change runtime type** in the top menu.  
2. Set **Hardware accelerator** to `L4 GPU`.  
3. Click **Save**.


**Step 2 – Set your API keys**  
Please set your API keys as environment variables to avoid exposing sensitive information in the code.

1. On the left sidebar, click the “🔑 Secrets” icon.  
2. Add the following secrets:  

   - **SEC_API_KEY** – Your SEC API key  
     - Obtain it from [SEC API](https://sec-api.io/profile) (used to retrieve 10-K reports).  

   - **FMP_API_KEY** – Your Financial Modeling Prep API key  
     - Obtain it from [Financial Modeling Prep](https://site.financialmodelingprep.com/developer/docs/dashboard) (used for target price data).
     -  **⚠️ Note:** According to the official documentation, when retrieving stock target price via the FMP API, only the following ticker symbols are available under the free subscription plan:
     > AAPL, TSLA, AMZN, MSFT, NVDA, GOOGL, META, NFLX, JPM, V, BAC, AMD, PYPL, DIS, T, PFE, COST, INTC, KO, TGT, NKE, SPY, BA, BABA, XOM, WMT, GE, CSCO, VZ, JNJ, CVX, PLTR, SQ, SHOP, SBUX, SOFI, HOOD, RBLX, SNAP, AMD, UBER, FDX, ABBV, ETSY, MRNA, LMT, GM, F, RIVN, LCID, CCL, DAL, UAL, AAL, TSM, SONY, ET, NOK, MRO, COIN, RIVN, SIRI, SOFI, RIOT, CPRX, PYPL, TGT, VWO, SPYG, NOK, ROKU, HOOD, VIAC, ATVI, BIDU, DOCU, ZM, PINS, TLRY, WBA, VIAC, MGM, NFLX, NIO, C, GS, WFC, ADBE, PEP, UNH, CARR, FUBO, HCA, TWTR, BILI, SIRI, VIAC, FUBO, RKT


   - **HF_TOKEN** - refers to your Hugging Face access token.
    - Visit: https://huggingface.co/settings/tokens

3. Enable **"Notebook Access"** for each secret on the left side.

No OpenAI key is needed: the notebook runs the free `qwen2.5:14b` model locally through Ollama.


**Step 3 – Run the entire notebook**  
Once the dependencies are installed and the runtime is set, go to **Runtime → Run all** to execute all cells.

⚠️ **When the “Restart session” windows appear, simply click Cancel** — it won’t affect the rest of the run. This is because:

1. `defusedxml` is a package preinstalled in Colab, but it gets reinstalled when installing openbb, which triggers the restart prompt.

2. `packaging` is a shared dependency used by multiple libraries, so its version may change during installation without causing issues.


---

## Code

### Environment Setup and Dependency Installation

In this notebook, we use the [OpenBB](https://github.com/OpenBB-finance/OpenBB) to conveniently fetch historical equity price data (e.g., Apple stock from Yahoo Finance).

#### Install openbb and test

In [1]:
# !pip install openbb
!pip install openbb==5.0.0 openbb-core==2.0.1 openbb-equity==2.0.0 openbb-yfinance==2.0.0 yfinance==1.7.0
from openbb import obb
obb.equity.price.historical("AAPL", provider="yfinance") # Return the history stock price

OBBject[T]

id: 06adf6ed-07ca-7255-8000-38d9311d3ea1
results: [{'date': datetime.date(2025, 10, 27), 'open': 264.8800048828125, 'high': ...
provider: yfinance
warnings: None
chart: None
extra: {'metadata': {'arguments': {'provider_choices': {'provider': 'yfinance'}, 's...

In [2]:
# torch/torchvision/torchaudio come preinstalled in Colab
!pip install reportlab yfinance matplotlib scrapy sec_api langchain umap-learn scikit-learn langchain_community tiktoken langchain-openai chromadb langchain-anthropic sentence-transformers

#### Install Ollama and download the model

Ollama runs the LLM on the Colab GPU. `OLLAMA_CONTEXT_LENGTH` raises the context window to 32k tokens, since each prompt includes the full 10-K Section 7 (Ollama's default would silently cut it off).

In [3]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time, requests

OLLAMA_MODEL = "qwen2.5:14b"
OLLAMA_URL = "http://localhost:11434/v1"

# Start the Ollama server in the background
subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_CONTEXT_LENGTH": "32768"},
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait until the server is up
for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        break
    except requests.ConnectionError:
        time.sleep(1)

!ollama pull {OLLAMA_MODEL}

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [4]:
# Get FinGPT's utils/ package and run from its folder
import os
if not os.path.exists("/content/FinGPT"):
    !git clone https://github.com/AI4Finance-Foundation/FinGPT.git /content/FinGPT
# langchain.prompts was removed in LangChain 1.x
!sed -i 's/^from langchain.prompts import ChatPromptTemplate$/from langchain_core.prompts import ChatPromptTemplate/' /content/FinGPT/fingpt/FinGPT_FinancialReportAnalysis/utils/rag.py
%cd /content/FinGPT/fingpt/FinGPT_FinancialReportAnalysis

Cloning into '/content/FinGPT'...
remote: Enumerating objects: 3298, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 3298 (delta 15), reused 14 (delta 11), pack-reused 3262 (from 3)
Receiving objects: 100% (3298/3298), 12.90 MiB | 21.27 MiB/s, done.
Resolving deltas: 100% (1845/1845), done.
/content/FinGPT/fingpt/FinGPT_FinancialReportAnalysis


In [5]:
import yfinance as yf
from matplotlib import pyplot as plt
from pandas.tseries.offsets import DateOffset
from sec_api import ExtractorApi
import requests
import pandas as pd
import json
import numpy as np
from openai import OpenAI
import os
from utils import get_earnings_transcript, Raptor
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings

from google.colab import userdata

#OS: I got an error: RuntimeError: dictionary changed size during iteration
# August 29: seems to have been fixed

/tmp/ipykernel_3323/2039637750.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings.sentence_transformer import (


### Configuration of API Keys, Model Selection, and Embedding Setup

This part uses Tesla (TSLA) as the ticker symbol and runs Qwen 2.5 (14B) locally through Ollama as the language model.

In [6]:
ticker_symbol = "TSLA"

In [8]:
sec_api_key = userdata.get('SEC_API_KEY')
fmp_api_key = userdata.get('FMP_API_KEY')

#### Model Selection

In [9]:
# define all the necessary variables
obb.user.credentials.fmp_api_key = fmp_api_key
USE_CACHE = True

# Served by the local Ollama server started above (fits on the L4's 24 GB with a 32k context)
llm = OLLAMA_MODEL

# embd = OpenAIEmbeddings()
# create the open-source embedding function
embd = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
# Ollama exposes an OpenAI-compatible API, so the OpenAI clients work against it
model = ChatOpenAI(temperature=0, model=llm, base_url=OLLAMA_URL, api_key="ollama")
rag_helper = Raptor(model, embd)

client = OpenAI(
    base_url=OLLAMA_URL,
    api_key='ollama', # required, but unused
)

/tmp/ipykernel_3323/2646030292.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embd = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Develop a tailored Financial Analysis Report

#### Test FMP API Connection

In [10]:
import requests
url = f"https://financialmodelingprep.com/stable/price-target-consensus?symbol=AAPL&apikey={fmp_api_key}"
response = requests.get(url)
print("Status Code:", response.status_code)
print("Response JSON:", response.json())


Status Code: 200
Response JSON: [{'symbol': 'AAPL', 'targetHigh': 400, 'targetLow': 245, 'targetConsensus': 340.14, 'targetMedian': 355}]


#### Fetching Latest SEC 10-K Filing

In [11]:
# "Develop a tailored Financial Analysis Report aligned with the user's individual needs, drawing insights from the supplied reference materials. Initiate interaction with the user to obtain essential specifics and resolve any ambiguities. Iteratively refine the Financial Analysis Report through consistent evaluations using the given evaluationRubric and gather user input to ensure the end product aligns with the users expectations. You MUST FOLLOW the rules in order.","role":"expert level accountant","department":"finance","task":"Create a Financial Analysis Report","task_description":"As an expert level accountant in the finance department, your task is to create a Financial Analysis Report that provides comprehensive insights into the financial performance and health of the company. The report should be accurate, detailed, and well-structured, showcasing key financial metrics, trends, and analysis. The finished work will be used by the management team and stakeholders to make informed decisions, identify areas of improvement, and assess the overall financial position of the company. Core success factors include attention to detail, analytical skills, and the ability to effectively communicate complex financial information. The success of the report will be measured by its ability to provide actionable recommendations and contribute to the improvement of financial decision-making processes."

class ReportAnalysis:
    def __init__(self, ticker_symbol):
        self.ticker_symbol = ticker_symbol
        self.stock = yf.Ticker(ticker_symbol)
        self.info = self.stock.info
        self.project_dir = f"projects/{ticker_symbol}/"
        self.cache_dir = f"projects/{ticker_symbol}/cache"
        os.makedirs(self.project_dir, exist_ok=True)
        os.makedirs(self.cache_dir, exist_ok=True)
        self.extractor = ExtractorApi(sec_api_key)
        self.report_address = self.get_sec_report_address()

        self.system_prompt = """
            Role: Expert Investor
            Department: Finance
            Primary Responsibility: Generation of Customized Financial Analysis Reports

            Role Description:
            As an Expert Investor within the finance domain, your expertise is harnessed to develop bespoke Financial Analysis Reports that cater to specific client requirements. This role demands a deep dive into financial statements and market data to unearth insights regarding a company's financial performance and stability. Engaging directly with clients to gather essential information and continuously refining the report with their feedback ensures the final product precisely meets their needs and expectations.

            Key Objectives:

            Analytical Precision: Employ meticulous analytical prowess to interpret financial data, identifying underlying trends and anomalies.
            Effective Communication: Simplify and effectively convey complex financial narratives, making them accessible and actionable to non-specialist audiences.
            Client Focus: Dynamically tailor reports in response to client feedback, ensuring the final analysis aligns with their strategic objectives.
            Adherence to Excellence: Maintain the highest standards of quality and integrity in report generation, following established benchmarks for analytical rigor.
            Performance Indicators:
            The efficacy of the Financial Analysis Report is measured by its utility in providing clear, actionable insights. This encompasses aiding corporate decision-making, pinpointing areas for operational enhancement, and offering a lucid evaluation of the company's financial health. Success is ultimately reflected in the report's contribution to informed investment decisions and strategic planning.
        """

    def get_target_price(self):

        url = f"https://financialmodelingprep.com/stable/price-target-consensus?symbol={ticker_symbol}&apikey={fmp_api_key}"


        # Send a GET request
        price_target = None
        response = requests.get(url)

        # Ensure the request was successful
        if response.status_code == 200:
            # Parse the JSON data
            data = response.json()

            # price_target = data[0]['priceTarget']
             # Get current price
            price_target = data[0]["targetConsensus"]
        else:
            print("Failed to retrieve data:", response.status_code)

        return price_target

    def get_stock_performance(self):
        def fetch_stock_data(ticker, period="1y"):
            stock = yf.Ticker(ticker)
            hist = stock.history(period=period)
            return hist['Close']

        target_close = fetch_stock_data(self.ticker_symbol)
        sp500_close = fetch_stock_data("^GSPC")

        # Calculate the rate of change
        company_change = (target_close - target_close.iloc[0]) / target_close.iloc[0] * 100
        sp500_change = (sp500_close - sp500_close.iloc[0]) / sp500_close.iloc[0] * 100

        # Calculate additional date points
        start_date = company_change.index.min()
        four_months = start_date + DateOffset(months=4)
        eight_months = start_date + DateOffset(months=8)
        end_date = company_change.index.max()

        # Prepare the plot
        plt.rcParams.update({'font.size': 20})  # Set a larger font size
        plt.figure(figsize=(14, 7))
        plt.plot(company_change.index, company_change, label=f'{self.info["shortName"]} Change %', color='blue')
        plt.plot(sp500_change.index, sp500_change, label='S&P 500 Change %', color='red')

        # Set the title and labels
        plt.title(f'{self.info["shortName"]} vs S&P 500 - Change % Over the Past Year')
        plt.xlabel('Date')
        plt.ylabel('Change %')

        # Set the x-axis tick labels
        plt.xticks([start_date, four_months, eight_months, end_date],
                [start_date.strftime('%Y-%m'),
                    four_months.strftime('%Y-%m'),
                    eight_months.strftime('%Y-%m'),
                    end_date.strftime('%Y-%m')])

        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        # plt.show()
        plot_path = f"{self.project_dir}/stock_performance.png"
        plt.savefig(plot_path)
        plt.close()
        return plot_path

    def get_pe_eps_performance(self):
        ss = self.get_income_stmt()
        eps = ss.loc['Diluted EPS', :]

        # Retrieve historical data for the past year
        historical_data = self.stock.history(period="5y")


        # Specify the dates and ensure they are in UTC timezone
        dates = pd.to_datetime(eps.index[::-1], utc=True)

        # To ensure we find the nearest trading day, we convert dates and look up the closest dat
        results = {}
        for date in dates:
            # If the specified date is not a trading day, use bfill and ffill to find the nearest closing price
            if date not in historical_data.index:
                close_price = historical_data.asof(date)
            else:
                close_price = historical_data.loc[date]

            results[date] = close_price['Close']


        pe = [p/e for p, e in zip(results.values(), eps.values[::-1])]
        dates = eps.index[::-1]
        eps = eps.values[::-1]

        # Create the figure and axis objects
        fig, ax1 = plt.subplots(figsize=(14, 7))
        plt.rcParams.update({'font.size': 20})  # Set a larger font size

        # Plot the PE ratio
        color = 'tab:blue'
        ax1.set_xlabel('Date')
        ax1.set_ylabel('PE Ratio', color=color)
        ax1.plot(dates, pe, color=color)
        ax1.tick_params(axis='y', labelcolor=color)
        ax1.grid(True)

        # Create a second axis sharing the x-axis with ax1
        ax2 = ax1.twinx()
        color = 'tab:red'
        ax2.set_ylabel('EPS', color=color)  # The label for the second y-axis
        ax2.plot(dates, eps, color=color)
        ax2.tick_params(axis='y', labelcolor=color)

        # Set the title and the rotation angle of the x-axis labels
        plt.title(f'{self.info["shortName"]} PE Ratios and EPS Over the Past 4 Years')
        plt.xticks(rotation=45)

        # Set the x-axis tick labels
        plt.xticks(dates, [d.strftime('%Y-%m') for d in dates])

        plt.tight_layout()
        # plt.show()
        plot_path = f"{self.project_dir}/pe_performance.png"
        plt.savefig(plot_path)
        plt.close()
        return plot_path

    def get_sec_report_address(self):
        address_json = f"{self.project_dir}/sec_report_address.json"
        if not os.path.exists(address_json):
            endpoint = f"https://api.sec-api.io?token={sec_api_key}"

            # The query to find 10-K filings for a specific company
            query = {
            "query": { "query_string": { "query": f"ticker:{self.ticker_symbol} AND formType:\"10-K\"" } },
            "from": "0",
            "size": "1",
            "sort": [{ "filedAt": { "order": "desc" } }]
            }

            # Making the request to the SEC API
            response = requests.post(endpoint, json=query)

            if response.status_code == 200:
                # Parsing the response
                filings = response.json()['filings']
                if filings:
                    # Assuming the latest 10-K filing is what we want
                    latest_10k_url = filings[0]
                    print(f"Latest 10-K report URL for {self.ticker_symbol}: {latest_10k_url}")
                else:
                    print(f"No 10-K filings found for {self.ticker_symbol}.")
            else:
                print("Failed to retrieve filings from SEC API.")

            with open(address_json, "w") as f:
                json.dump(latest_10k_url, f)
        else:
            with open(address_json, "r") as f:
                latest_10k_url = json.load(f)

        return latest_10k_url['linkToFilingDetails']

    def get_key_data(self):
        # Fetch historical market data for the past 6 months
        hist = self.stock.history(period="6mo")

        # Retrieve other related information
        info = self.info
        close_price = hist['Close'].iloc[-1]

        # Calculate the average daily trading volume
        avg_daily_volume_6m = hist['Volume'].mean()

        # Print the result
        # print(f"Over the past 6 months, the average daily trading volume for {ticker_symbol} was: {avg_daily_volume_6m:.2f}")
        result = {
            f"6m avg daily val ({info['currency']}mn)": "{:.2f}".format(avg_daily_volume_6m/1e6),
            f"Closing Price ({info['currency']})": "{:.2f}".format(close_price),
            f"Market Cap ({info['currency']}mn)": "{:.2f}".format(info['marketCap']/1e6),
            f"52 Week Price Range ({info['currency']})": f"{info['fiftyTwoWeekLow']} - {info['fiftyTwoWeekHigh']}",
            f"BVPS ({info['currency']})": info['bookValue']
        }
        return result

    def get_company_info(self):
        info = self.info
        result = {
            "Company Name": info['shortName'],
            "Industry": info['industry'],
            "Sector": info['sector'],
            "Country": info['country'],
            "Website": info['website']
        }
        return result

    def get_income_stmt(self):
        income_stmt = self.stock.financials
        return income_stmt

    def get_balance_sheet(self):
        balance_sheet = self.stock.balance_sheet
        return balance_sheet

    def get_cash_flow(self):
        cash_flow = self.stock.cashflow
        return cash_flow

    def get_analyst_recommendations(self):
        recommendations = self.stock.recommendations
        row_0 = recommendations.iloc[0, 1:]  # Exclude 'period' column

        # Find the maximum voting result
        max_votes = row_0.max()
        majority_voting_result = row_0[row_0 == max_votes].index.tolist()

        return majority_voting_result[0], max_votes

    def get_earnings(self, quarter, year):
        earnings = get_earnings_transcript(quarter, self.ticker_symbol, year)
        return earnings

    def get_10k_section(self, section):
        """
            Get 10-K reports from SEC EDGAR
        """
        if section not in [1, "1A", "1B", 2, 3, 4, 5, 6, 7, "7A", 8, 9, "9A", "9B", 10, 11, 12, 13, 14, 15]:
            raise ValueError("Section must be in [1, 1A, 1B, 2, 3, 4, 5, 6, 7, 7A, 8, 9, 9A, 9B, 10, 11, 12, 13, 14, 15]")

        section = str(section)
        os.makedirs(f"{self.project_dir}/10k", exist_ok=True)

        report_name = f"{self.project_dir}/10k/section_{section}.txt"

        if USE_CACHE and os.path.exists(report_name):
            with open(report_name, "r") as f:
                section_text = f.read()
        else:
            section_text = self.extractor.get_section(self.report_address, section, "text")

            with open(report_name, "w") as f:
                f.write(section_text)

        return section_text

    def get_10k_rag(self, section):
        # Now, use all_texts to build the vectorstore with Chroma
        vector_dir = f"{self.cache_dir}/section_{section}_vectorstore"
        if USE_CACHE and os.path.exists(vector_dir):
            vectorstore = Chroma(persist_directory=vector_dir, embedding_function=embd)
            vectorstore.get()
        else:
            section_text = self.get_10k_section(section)
            all_texts = rag_helper.text_spliter(section_text, chunk_size_tok=2000, level=1, n_levels=3)

            vectorstore = Chroma.from_texts(texts=all_texts, embedding=embd, persist_directory=vector_dir)
            vectorstore.persist()

        retriever = vectorstore.as_retriever()

        # Prompt
        prompt = ChatPromptTemplate.from_messages([("human",
            "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. "
            "If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n"
            "Question: {question} \nContext: {context} \nAnswer:")])

        # Chain
        rag_chain = (
            # {"context": retriever | format_docs, "question": RunnablePassthrough()}
            {"context": retriever, "question": RunnablePassthrough()}
            | prompt
            | model
            | StrOutputParser()
        )

        # Question
        # rag_chain.invoke("What is the profit of the company. you should not say you don't know because all the required information is in the context")
        # rag_chain.invoke("Analyse the income statement of the company for the year 2023")
        return rag_chain

    def analyze_income_stmt(self):
        cache_answer = f"{self.project_dir}/income_stmt_analysis.txt"
        if USE_CACHE and os.path.exists(cache_answer):
            with open(cache_answer, "r") as f:
                answer = f.read()
        else:
            income_stmt = self.get_income_stmt()
            df_string = "Income statement:" + income_stmt.to_string().strip()

            question = "Embark on a thorough analysis of the company's income statement for the current fiscal year, focusing on revenue streams, cost of goods sold, operating expenses, and net income to discern the operational performance and profitability. Examine the gross profit margin to understand the cost efficiency, operating margin for operational effectiveness, and net profit margin to assess overall profitability. Compare these financial metrics against historical data to identify growth patterns, profitability trends, and operational challenges. Conclude with a strategic overview of the company's financial health, offering insights into revenue growth sustainability and potential areas for cost optimization and profit maximization in a single paragraph. Less than 130 words."

            answer = self.ask_question(question, 7, df_string, use_rag=False)
            with open(cache_answer, "w") as f:
                f.write(answer)
        return answer

    def analyze_balance_sheet(self):
        cache_answer = f"{self.project_dir}/balance_sheet_analysis.txt"
        if USE_CACHE and os.path.exists(cache_answer):
            with open(cache_answer, "r") as f:
                answer = f.read()
        else:
            balance_sheet = self.get_balance_sheet()
            df_string = "Balance sheet:" + balance_sheet.to_string().strip()

            question = "Delve into a detailed scrutiny of the company's balance sheet for the most recent fiscal year, pinpointing the structure of assets, liabilities, and shareholders' equity to decode the firm's financial stability and operational efficiency. Focus on evaluating the liquidity through current assets versus current liabilities, the solvency via long-term debt ratios, and the equity position to gauge long-term investment potential. Contrast these metrics with previous years' data to highlight financial trends, improvements, or deteriorations. Finalize with a strategic assessment of the company's financial leverage, asset management, and capital structure, providing insights into its fiscal health and future prospects in a single paragraph. Less than 130 words."

            answer = self.ask_question(question, 7, df_string, use_rag=False)
            with open(cache_answer, "w") as f:
                f.write(answer)
        return answer

    def analyze_cash_flow(self):
        cache_answer = f"{self.project_dir}/cash_flow_analysis.txt"
        if USE_CACHE and os.path.exists(cache_answer):
            with open(cache_answer, "r") as f:
                answer = f.read()
        else:
            cash_flow = self.get_cash_flow()
            df_string = "Balance sheet:" + cash_flow.to_string().strip()

            question = "Dive into a comprehensive evaluation of the company's cash flow for the latest fiscal year, focusing on cash inflows and outflows across operating, investing, and financing activities. Examine the operational cash flow to assess the core business profitability, scrutinize investing activities for insights into capital expenditures and investments, and review financing activities to understand debt, equity movements, and dividend policies. Compare these cash movements to prior periods to discern trends, sustainability, and liquidity risks. Conclude with an informed analysis of the company's cash management effectiveness, liquidity position, and potential for future growth or financial challenges in a single paragraph. Less than 130 words."

            answer = self.ask_question(question, 7, df_string, use_rag=False)
            with open(cache_answer, "w") as f:
                f.write(answer)
        return answer

    def financial_summarization(self):
        income_stmt_analysis = self.analyze_income_stmt()
        balance_sheet_analysis = self.analyze_balance_sheet()
        cash_flow_analysis = self.analyze_cash_flow()

        cache_answer = f"{self.project_dir}/financial_summarization.txt"
        if USE_CACHE and os.path.exists(cache_answer):
            with open(cache_answer, "r") as f:
                answer = f.read()
        else:
            question = f"Income statement analysis: {income_stmt_analysis}, \
            Balance sheet analysis: {balance_sheet_analysis}, \
            Cash flow analysis: {cash_flow_analysis}, \
            Synthesize the findings from the in-depth analysis of the income statement, balance sheet, and cash flow for the latest fiscal year. Highlight the core insights regarding the company's operational performance, financial stability, and cash management efficiency. Discuss the interrelations between revenue growth, cost management strategies, and their impact on profitability as revealed by the income statement. Incorporate the balance sheet's insights on financial structure, liquidity, and solvency to provide a comprehensive view of the company's financial health. Merge these with the cash flow analysis to illustrate the company's liquidity position, investment activities, and financing strategies. Conclude with a holistic assessment of the company's fiscal health, identifying strengths, potential risks, and strategic opportunities for growth and stability. Offer recommendations to address identified challenges and capitalize on the opportunities to enhance shareholder value in a single paragraph. Less than 150 words."

            answer = self.ask_question(question, 7, use_rag=False)
            with open(cache_answer, "w") as f:
                f.write(answer)
        return {"Income Statement Analysis": income_stmt_analysis, "Balance Sheet Analysis": balance_sheet_analysis, "Cash Flow Analysis": cash_flow_analysis, "Financial Summary": answer}


    def ask_question(self, question, section, table_str=None, use_rag=False):
        if use_rag:
            rag_chain = self.get_10k_rag(section)
            if table_str:
                prompt = f"{self.system_prompt}\n\n{table_str}\n\nQuestion: {question}"
            else:
                prompt = f"{self.system_prompt}\n\nQuestion: {question}"
            answer = rag_chain.invoke(prompt)
        else:
            # Send a request to the local Ollama server using the specified model
            section_text = self.get_10k_section(7)
            if table_str:
                prompt = f"{self.system_prompt}\n\n{table_str}\n\nResource: {section_text}\n\nQuestion: {question}"
            else:
                prompt = f"{self.system_prompt}\n\nResource: {section_text}\n\nQuestion: {question}"

            chat_completion = client.chat.completions.create(
                messages=[
                    {
                        "role": "user",
                        "content": prompt.strip(),
                    }
                ],
                # model="gpt-4-1106-preview",
                model=llm,
                temperature = 0,
                max_tokens = 500,
                # response_format={ "type": "json_object" },
            )
            answer = chat_completion.choices[0].message.content

        return answer


ra = ReportAnalysis(ticker_symbol)

Latest 10-K report URL for TSLA: {'ticker': 'TSLA', 'formType': '10-K/A', 'accessionNo': '0001104659-26-053166', 'cik': '1318605', 'companyNameLong': 'Tesla, Inc. (Filer)', 'companyName': 'Tesla, Inc.', 'linkToFilingDetails': 'https://www.sec.gov/Archives/edgar/data/1318605/000110465926053166/tm2611837d1_10ka.htm', 'description': 'Form 10-K/A - Annual report [Section 13 and 15(d), not S-K Item 405]: [Amend]', 'linkToTxt': 'https://www.sec.gov/Archives/edgar/data/1318605/000110465926053166/0001104659-26-053166.txt', 'filedAt': '2026-04-30T17:13:17-04:00', 'documentFormatFiles': [{'sequence': '1', 'size': '691471', 'documentUrl': 'https://www.sec.gov/ix?doc=/Archives/edgar/data/1318605/000110465926053166/tm2611837d1_10ka.htm', 'description': 'FORM 10-K/A', 'type': '10-K/A'}, {'sequence': '2', 'size': '2917', 'documentUrl': 'https://www.sec.gov/Archives/edgar/data/1318605/000110465926053166/tm2611837d1_ex31-3.htm', 'description': 'EXHIBIT 31.3', 'type': 'EX-31.3'}, {'sequence': '3', 'size

In [12]:
answer = ra.financial_summarization()

### Build the pdf Report

The default report path is FinGPT/fingpt/FinGPT_FinancialReportAnalysis/projects/TSLA/TSLA_report.pdf

In [13]:
from reportlab.lib import colors
from reportlab.lib import pagesizes
from reportlab.platypus import SimpleDocTemplate, Frame, Paragraph, Image, PageTemplate, FrameBreak, Spacer, Table, TableStyle, NextPageTemplate, PageBreak
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT


# 2. Create the PDF and insert images
# Page setup
page_width, page_height = pagesizes.A4
left_column_width = page_width * 2/3
right_column_width = page_width - left_column_width
margin = 4

# Create the PDF document path
pdf_path = os.path.join(ra.project_dir, f"{ticker_symbol}_report.pdf")
doc = SimpleDocTemplate(pdf_path, pagesize=pagesizes.A4)

# Define two column frames
frame_left = Frame(margin, margin, left_column_width-margin*2, page_height-margin*2, id='left')
frame_right = Frame(left_column_width, margin, right_column_width-margin*2, page_height-margin*2, id='right')

# single_frame = Frame(margin, margin, page_width-margin*2, page_height-margin*2, id='single')
# single_column_layout = PageTemplate(id='OneCol', frames=[single_frame])

left_column_width_p2 = (page_width-margin*3) // 2
right_column_width_p2 = left_column_width_p2
frame_left_p2 = Frame(margin, margin, left_column_width_p2-margin*2, page_height-margin*2, id='left')
frame_right_p2 = Frame(left_column_width_p2, margin, right_column_width_p2-margin*2, page_height-margin*2, id='right')

# Create the PageTemplate and add it to the document
page_template = PageTemplate(id='TwoColumns', frames=[frame_left, frame_right])
page_template_p2 = PageTemplate(id='TwoColumns_p2', frames=[frame_left_p2, frame_right_p2])
doc.addPageTemplates([page_template, page_template_p2])

styles = getSampleStyleSheet()

# Define custom styles
custom_style = ParagraphStyle(
    name="Custom",
    parent=styles['Normal'],
    fontName="Helvetica",
    fontSize=10,
    # leading=15,
    alignment=TA_JUSTIFY,
)

title_style = ParagraphStyle(
    name="TitleCustom",
    parent=styles['Title'],
    fontName="Helvetica-Bold",
    fontSize=16,
    leading=20,
    alignment=TA_LEFT,
    spaceAfter=10,
)

subtitle_style = ParagraphStyle(
    name="Subtitle",
    parent=styles['Heading2'],
    fontName="Helvetica-Bold",
    fontSize=14,
    leading=12,
    alignment=TA_LEFT,
    spaceAfter=6,
)

# Prepare content for the left and right columns
content = []
# Title
content.append(Paragraph(f"Equity Research Report: {ra.get_company_info()['Company Name']}", title_style))

# Subtitle
content.append(Paragraph("Income Statement Analysis", subtitle_style))
content.append(Paragraph(answer['Income Statement Analysis'], custom_style))

content.append(Paragraph("Balance Sheet Analysis", subtitle_style))
content.append(Paragraph(answer['Balance Sheet Analysis'], custom_style))

content.append(Paragraph("Cashflow Analysis", subtitle_style))
content.append(Paragraph(answer['Cash Flow Analysis'], custom_style))

content.append(Paragraph("Summarization", subtitle_style))
content.append(Paragraph(answer['Financial Summary'], custom_style))


content.append(FrameBreak())  # Used to switch from the left column to the right column

table_style = TableStyle([
    ('BACKGROUND', (0, 0), (-1, -1), colors.white),
    ('BACKGROUND', (0, 0), (-1, 0), colors.white),
    ('FONT', (0, 0), (-1, -1), 'Helvetica', 8),
    ('FONT', (0, 0), (-1, 0), 'Helvetica-Bold', 12),
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    # Left-align the first column
    ('ALIGN', (0,1), (0,-1), 'LEFT'),
    # Right-align the second column
    ('ALIGN', (1,1), (1,-1), 'RIGHT'),
    # Add a horizontal line below the header row
    ('LINEBELOW', (0,0), (-1,0), 2, colors.black),
])
full_length = right_column_width-2*margin

rating, _ = ra.get_analyst_recommendations()
target_price = ra.get_target_price()
if target_price is not None:
    data = [
        ["Rating:", rating.upper()],
        ["Target Price:", f"{target_price:.2f}"]
    ]
else:
    data = [["Rating:", rating.upper()]]
col_widths = [full_length//3*2, full_length//3]
table = Table(data, colWidths=col_widths)
table.setStyle(table_style)
content.append(table)

# content.append(Paragraph("", custom_style))
content.append(Spacer(1, 0.15*inch))
key_data = ra.get_key_data()
# Table data
data = [["Key data", ""]]
data += [
    [k, v] for k, v in key_data.items()
]
col_widths = [full_length//3*2, full_length//3]
table = Table(data, colWidths=col_widths)
table.setStyle(table_style)
content.append(table)


# Add Matplotlib images to the right column

# Historical share price
data = [["Share Performance"]]
col_widths = [full_length]
table = Table(data, colWidths=col_widths)
table.setStyle(table_style)
content.append(table)

plot_path = ra.get_stock_performance()
width = right_column_width
height = width//2
content.append(Image(plot_path, width=width, height=height))

# Historical PE ratio and EPS
data = [["PE & EPS"]]
col_widths = [full_length]
table = Table(data, colWidths=col_widths)
table.setStyle(table_style)
content.append(table)

plot_path = ra.get_pe_eps_performance()
width = right_column_width
height = width//2
content.append(Image(plot_path, width=width, height=height))


# Start a new page
content.append(NextPageTemplate('TwoColumns_p2'))
content.append(PageBreak())

table_style2 = TableStyle([
    ('BACKGROUND', (0, 0), (-1, -1), colors.white),
    ('BACKGROUND', (0, 0), (-1, 0), colors.white),
    ('FONT', (0, 0), (-1, -1), 'Helvetica', 6),
    ('FONT', (0, 0), (-1, 0), 'Helvetica-Bold', 10),
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    # Left-align the first column
    ('ALIGN', (0,1), (0,-1), 'LEFT'),
    # Right-align the second column
    ('ALIGN', (1,1), (1,-1), 'RIGHT'),
    # Add a horizontal line below the header row
    ('LINEBELOW', (0,0), (-1,0), 2, colors.black),
    # Add a horizontal line below the last row of the table
    ('LINEBELOW', (0,-1), (-1,-1), 2, colors.black),
])


# Use single-column layout for the second and subsequent pages
df = ra.get_income_stmt()
df = df[df.columns[:3]]
def convert_if_money(value):
    if np.abs(value) >= 1000000:
        return value / 1000000
    else:
        return value

# Apply the conversion function to each column of the DataFrame
df = df.map(convert_if_money)

df.columns = [col.strftime('%Y') for col in df.columns]
df.reset_index(inplace=True)
currency = ra.info['currency']
df.rename(columns={'index': f'FY ({currency} mn)'}, inplace=True)  # 可选：重命名索引列为“序号”
table_data = [["Income Statement"]]
table_data += [df.columns.to_list()] + df.values.tolist()

table = Table(table_data)
table.setStyle(table_style2)
content.append(table)

content.append(FrameBreak())  # Used to switch from the left column to the right column

df = ra.get_cash_flow()
df = df[df.columns[:3]]

df = df.map(convert_if_money)

df.columns = [col.strftime('%Y') for col in df.columns]
df.reset_index(inplace=True)
currency = ra.info['currency']
df.rename(columns={'index': f'FY ({currency} mn)'}, inplace=True)  # 可选：重命名索引列为“序号”
table_data = [["Cash Flow Sheet"]]
table_data += [df.columns.to_list()] + df.values.tolist()

table = Table(table_data)
table.setStyle(table_style2)
content.append(table)
# content.append(Paragraph('This is a single column on the second page', custom_style))
# content.append(Spacer(1, 0.2*inch))
# content.append(Paragraph('More content in the single column.', custom_style))

# Build the PDF document
doc.build(content)
print(pdf_path)

projects/TSLA/TSLA_report.pdf


In [14]:
!pip list

Package                                  Version
---------------------------------------- -------------------
a2wsgi                                   1.10.10
absl-py                                  2.0.0
accelerate                               1.15.0
access                                   1.1.10.post3
affine                                   3.0.1
aiofile                                  3.12.3
aiofiles                                 25.1.0
aiohappyeyeballs                         2.7.1
aiohttp                                  3.14.5
aiohttp-client-cache                     0.15.0
aiohttp-retry                            2.9.1
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.12.1
altair                                   5.5.0
annotated-doc              

## Declaration

This section is adapted from the [reportanalysis.ipynb](https://github.com/AI4Finance-Foundation/FinGPT/blob/master/fingpt/FinGPT_FinancialReportAnalysis/reportanalysis.ipynb) notebook in the **FinGPT/FinGPT_FinancialReportAnalysis** repository.